In [1]:
# CELL 1: Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import json
import warnings
warnings.filterwarnings('ignore')
from transformers import ViTForImageClassification, ViTFeatureExtractor

# Configuration
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: mps


## Load Database

In [2]:
from dataset import CarDamageDataset

data = torch.load("../../../development/database/cars_damage_dataset/database/data_preparation_outputs.pth", weights_only=False)
train_loader = data['train_loader']
val_loader = data['val_loader']
label_encoder = data['label_encoder']
class_mapping = data['class_mapping']

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Using device: mps
Training batches: 21
Validation batches: 6
Number of classes: 3


## Setup Vision Transformer

In [3]:
model_name = "google/vit-base-patch16-224-in21k"
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_),
    ignore_mismatched_sizes=True
)
model = model.to(DEVICE)

print(f"Model loaded: {model_name}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: google/vit-base-patch16-224-in21k
Number of classes: 3


## Setup Optimizer and Scheduler

In [4]:
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
criterion = nn.CrossEntropyLoss()

print("Setup complete. Ready to train the model.")

Setup complete. Ready to train the model.


## Training function

In [5]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(pixel_values=images)